In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/home4/s6019595/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
model_path = "./L1-Qwen-1.5B-Exact"
model_LCPO = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.66it/s]


In [77]:
def run_inference(model_LCPO, tokenizer, device, prompt):
    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_LCPO(**inputs)
    past_key_values = outputs.past_key_values
    generated = inputs.input_ids
    # Generate until EOS token
    while True:
        with torch.no_grad():
            out = model_LCPO(
                input_ids=generated[:, -1:],  # Start from laft logit of already processed prompt
                past_key_values=past_key_values,
                use_cache=True,
            )
        logits = out.logits[:, -1, :]
        past_key_values = out.past_key_values
        next_token = torch.argmax(logits, dim=-1)
        generated = torch.cat([generated, next_token.unsqueeze(-1)], dim=-1)

        # EOS check
        if next_token.item() == tokenizer.eos_token_id:
            print("Stopped: EOS emitted")
            break

    full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return full_text, generated.shape[1]

In [98]:
target_tokens = 300
prompt = f"that regulates the trade-off between generating the correct answer and meeting the target length. In practice, a lower value of α prioritizes correctness when it is critical, whereas a higher value enforces stricter adherence to the length constraint. Notably, the reward function serves a dual purpose: (a) it encourages the model to produce correct answers while implicitly favoring concise reasoning traces when shorter outputs are requested, and (b) it consistently motivates the model to match the prescribed target length even when a correct answer could be generated with fewer tokens. We refer to the model trained with this objective as L1-Exact.Solve: What is 23 * 19 * 30?, Think for {target_tokens} tokens."
full_text, generated_tokens = run_inference(model_LCPO, tokenizer, device, prompt)
print(f"Actual | Expected number of tokens: {generated_tokens} | {target_tokens}")
print("====================================")
print(f"All generated text:\n{full_text}")

Stopped: EOS emitted
Actual | Expected number of tokens: 663 | 300
All generated text:
that regulates the trade-off between generating the correct answer and meeting the target length. In practice, a lower value of α prioritizes correctness when it is critical, whereas a higher value enforces stricter adherence to the length constraint. Notably, the reward function serves a dual purpose: (a) it encourages the model to produce correct answers while implicitly favoring concise reasoning traces when shorter outputs are requested, and (b) it consistently motivates the model to match the prescribed target length even when a correct answer could be generated with fewer tokens. We refer to the model trained with this objective as L1-Exact.Solve: What is 23 * 19 * 30?, Think for 300 tokens. Wait, maybe I'm getting confused. Let me try to parse this.

Okay, so the user is talking about a model that's trained to solve math problems, specifically 23 * 19 * 30. They mention that the model's traini

In [91]:
#tokenizer returns dictionaty of 2 elements , input_ids which contains the ids of the tokens in the string and attention mask for each input_is
inputs = tokenizer(prompt, return_tensors="pt").to(device)
print(tokenizer.convert_ids_to_tokens(inputs.input_ids[0].tolist()))
print(inputs.keys())
print(inputs.input_ids)
print(inputs.attention_mask)

['<｜begin▁of▁sentence｜>', 'S', 'olve', ':', 'ĠWhat', 'Ġis', 'Ġ', '2', '3', 'Ġ*', 'Ġ', '1', '9', 'Ġ*', 'Ġ', '3', '0', '?,', 'ĠThink', 'Ġfor', 'Ġ', '5', '0', 'Ġtokens', '.']
dict_keys(['input_ids', 'attention_mask'])
tensor([[151646,     50,   3948,     25,   3555,    374,    220,     17,     18,
            353,    220,     16,     24,    353,    220,     18,     15,  12622,
          21149,    369,    220,     20,     15,  11211,     13]],
       device='cuda:0')
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]], device='cuda:0')
